# 10 pi_0 从权限门控到正式训练

            pi_0 的模型加载、gated 权限和显存压力都更高。这里先把权限、数据、模型构造和 2-step 反向传播逐项过门，再启动正式训练。

            这一节从 LeRobot 数据检查开始，依次完成 2-step smoke、正式训练、ROCm 资源观察和 checkpoint 定位。所有长任务默认关闭，先确认生成的配置，再显式打开运行开关。


In [1]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys


def find_topic_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "assets" / "metrics_snapshot.json").exists():
            return candidate
    raise RuntimeError("请从 AMD ROCm 专题目录或 notebooks 子目录启动 Jupyter。")


TOPIC_ROOT = find_topic_root()
ASSET_DIR = TOPIC_ROOT / "assets"
NOTEBOOK_DIR = TOPIC_ROOT / "notebooks"
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/path/to/every-embodied/mujoco_pnp"))
DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/path/to/datasets/every_embodied"))
OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", TOPIC_ROOT / "outputs"))
MODEL_ROOT = Path(os.environ.get("MODEL_ROOT", PROJECT_ROOT / "ckpt"))

print("TOPIC_ROOT =", TOPIC_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT =", DATA_ROOT)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("MODEL_ROOT =", MODEL_ROOT)


TOPIC_ROOT = /home/aup/amd-notebook-work/course
PROJECT_ROOT = /path/to/every-embodied/mujoco_pnp
DATA_ROOT = /path/to/datasets/every_embodied
OUTPUT_ROOT = /home/aup/amd-notebook-work/course/outputs
MODEL_ROOT = /path/to/every-embodied/mujoco_pnp/ckpt


In [2]:
try:
    from IPython.display import Image, Markdown, display
except Exception:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)

    def Image(filename=None, width=None):
        return f"[image] {filename}"


def show_asset(filename, width=960):
    path = ASSET_DIR / filename
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f"缺少素材：{path}")


def md_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(str(x) for x in row) + " |")
    display(Markdown("\n".join(lines)))


In [3]:
import shlex

try:
    import yaml
except ImportError as exc:
    raise RuntimeError("当前环境缺少 PyYAML，请先执行 pip install pyyaml。") from exc


def require_project_layout():
    required = [
        PROJECT_ROOT / "train_model.py",
        PROJECT_ROOT / "env_config.py",
        PROJECT_ROOT / "asset" / "example_scene_y2.xml",
        PROJECT_ROOT / "mujoco_env" / "y_env2.py",
    ]
    missing = [path for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "PROJECT_ROOT 不是 04mujoco 教程目录，缺少：\n"
            + "\n".join(str(path) for path in missing)
        )
    return True


def dataset_report(dataset_root):
    dataset_root = Path(dataset_root)
    info_path = dataset_root / "meta" / "info.json"
    tasks_path = dataset_root / "meta" / "tasks.jsonl"
    if not info_path.exists():
        raise FileNotFoundError(f"找不到 LeRobot 元数据：{info_path}")
    info = json.loads(info_path.read_text(encoding="utf-8"))
    tasks = []
    if tasks_path.exists():
        tasks = [
            json.loads(line)["task"]
            for line in tasks_path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]
    features = info.get("features", {})
    rows = [
        ("repo_id", info.get("repo_id", "")),
        ("episodes", info.get("total_episodes", 0)),
        ("frames", info.get("total_frames", 0)),
        ("fps", info.get("fps", "")),
        ("state shape", features.get("observation.state", {}).get("shape")),
        ("action shape", features.get("action", {}).get("shape")),
        ("tasks", " / ".join(tasks)),
    ]
    md_table(["数据项", "读取结果"], rows)
    return info, tasks


def make_training_config(
    policy_type,
    dataset_repo_id,
    dataset_root,
    output_dir,
    steps,
    batch_size,
    chunk_size,
    n_action_steps,
    save_freq,
    seed=42,
):
    return {
        "dataset": {"repo_id": dataset_repo_id, "root": str(Path(dataset_root))},
        "policy": {
            "type": policy_type,
            "chunk_size": int(chunk_size),
            "n_action_steps": int(n_action_steps),
            "device": "cuda",
        },
        "save_checkpoint": True,
        "output_dir": str(Path(output_dir)),
        "batch_size": int(batch_size),
        "job_name": Path(output_dir).name,
        "resume": False,
        "seed": int(seed),
        "num_workers": 4,
        "steps": int(steps),
        "eval_freq": 0,
        "log_freq": max(1, min(50, int(steps))),
        "save_freq": int(save_freq),
        "use_policy_training_preset": True,
        "wandb": {
            "enable": False,
            "project": f"every_embodied_{policy_type}",
            "entity": None,
            "disable_artifact": True,
        },
    }


def write_training_config(name, config):
    config_dir = OUTPUT_ROOT / "configs"
    config_dir.mkdir(parents=True, exist_ok=True)
    path = config_dir / f"{name}.yaml"
    path.write_text(
        yaml.safe_dump(config, allow_unicode=True, sort_keys=False),
        encoding="utf-8",
    )
    print("已写出配置：", path)
    return path


def training_command(config_path):
    return [
        sys.executable,
        str(PROJECT_ROOT / "train_model.py"),
        "--config_path",
        str(Path(config_path)),
    ]


def run_training(config_path, enabled=False):
    command = training_command(config_path)
    print("$", shlex.join(command))
    if not enabled:
        print("当前只预览命令。确认配置后，把对应 RUN_* 开关改为 True。")
        return None
    require_project_layout()
    env = os.environ.copy()
    env["PYTHONPATH"] = f"{PROJECT_ROOT}:{env.get('PYTHONPATH', '')}"
    return subprocess.run(command, cwd=PROJECT_ROOT, env=env, check=True)


def show_rocm_resources():
    command = ["rocm-smi", "--showuse", "--showmemuse", "--showtemp"]
    print("$", shlex.join(command))
    if shutil.which(command[0]) is None:
        print("未找到 rocm-smi。确认当前机器是否安装 ROCm。")
        return
    result = subprocess.run(command, check=False, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout.rstrip())
    if result.stderr:
        print(result.stderr.rstrip(), file=sys.stderr)


## Checkpoint 1：确认项目和训练数据


In [4]:
if os.environ.get("AMD_RUN_FULL", "0") == "1":
    require_project_layout()
else:
    print("阅读模式：外部采集与训练工程在启用完整实验时检查。")
print("train_model.py =", PROJECT_ROOT / "train_model.py")
print("训练数据 =", Path(os.environ.get("TRAIN_DATA_ROOT", DATA_ROOT / "omy_pnp_language")))


阅读模式：外部采集与训练工程在启用完整实验时检查。
train_model.py = /path/to/every-embodied/mujoco_pnp/train_model.py
训练数据 = /path/to/datasets/every_embodied/omy_pnp_language


In [5]:
candidate = Path(os.environ.get("TRAIN_DATA_ROOT", DATA_ROOT / "omy_pnp_language"))
if (candidate / "meta" / "info.json").exists():
    dataset_report(candidate)
else:
    print("训练数据还没准备好：", candidate)
    print("先完成 07_data_collection_and_audit.ipynb，或设置 TRAIN_DATA_ROOT 指向已有 LeRobot 数据集。")


训练数据还没准备好： /path/to/datasets/every_embodied/omy_pnp_language
先完成 07_data_collection_and_audit.ipynb，或设置 TRAIN_DATA_ROOT 指向已有 LeRobot 数据集。


## Checkpoint 2：检查 Hugging Face gated 权限


In [6]:
token_present = bool(os.environ.get("HF_TOKEN"))
print("HF_TOKEN 已注入当前进程：", token_present)
if not token_present:
    print("请在终端私密设置 HF_TOKEN，再重新启动 Jupyter kernel。不要把 token 写进 Notebook。")


HF_TOKEN 已注入当前进程： False
请在终端私密设置 HF_TOKEN，再重新启动 Jupyter kernel。不要把 token 写进 Notebook。


这里只检查环境变量是否存在，不打印 token。pi_0 训练还需要账号已经获得 PaliGemma 等 gated 权重的访问权限。


## Checkpoint 3：生成 smoke 与正式训练配置


In [7]:
MODEL_TYPE = 'pi0'
DATASET_REPO_ID = os.environ.get("DATASET_REPO_ID", "datawhale_eai_pnp_language")
TRAIN_DATA_ROOT = Path(
    os.environ.get("TRAIN_DATA_ROOT", DATA_ROOT / "omy_pnp_language")
).expanduser()

RUN_SMOKE = False
RUN_FULL_TRAIN = False

SMOKE_OUTPUT = MODEL_ROOT / f"{MODEL_TYPE}_rocm_smoke"
FULL_OUTPUT = MODEL_ROOT / f"{MODEL_TYPE}_rocm_full"

smoke_config = make_training_config(
    policy_type=MODEL_TYPE,
    dataset_repo_id=DATASET_REPO_ID,
    dataset_root=TRAIN_DATA_ROOT,
    output_dir=SMOKE_OUTPUT,
    steps=2,
    batch_size=min(4, 4),
    chunk_size=5,
    n_action_steps=5,
    save_freq=2,
)
full_config = make_training_config(
    policy_type=MODEL_TYPE,
    dataset_repo_id=DATASET_REPO_ID,
    dataset_root=TRAIN_DATA_ROOT,
    output_dir=FULL_OUTPUT,
    steps=20000,
    batch_size=4,
    chunk_size=5,
    n_action_steps=5,
    save_freq=max(1, 20000 // 2),
)

smoke_config_path = write_training_config(f"{MODEL_TYPE}_smoke", smoke_config)
full_config_path = write_training_config(f"{MODEL_TYPE}_full", full_config)
print("smoke output =", SMOKE_OUTPUT)
print("full output =", FULL_OUTPUT)


已写出配置： /home/aup/amd-notebook-work/course/outputs/configs/pi0_smoke.yaml
已写出配置： /home/aup/amd-notebook-work/course/outputs/configs/pi0_full.yaml
smoke output = /path/to/every-embodied/mujoco_pnp/ckpt/pi0_rocm_smoke
full output = /path/to/every-embodied/mujoco_pnp/ckpt/pi0_rocm_full


配置文件会写到 `$OUTPUT_ROOT/configs`，checkpoint 写到 `$MODEL_ROOT`。这两个目录应放在容量充足的磁盘，不要写进 Git 仓库。


## Checkpoint 4：先跑 2-step smoke


In [8]:
show_rocm_resources()
run_training(smoke_config_path, enabled=RUN_SMOKE)


$ rocm-smi --showuse --showmemuse --showtemp


============================ ROCm System Management Interface ============================
====================================== Temperature =======================================
GPU[0]		: Temperature (Sensor edge) (C): 35.0
=================================== % time GPU is busy ===================================
GPU[0]		: GPU use (%): 0
=================================== Current Memory Use ===================================
GPU[0]		: GPU Memory Allocated (VRAM%): 0
GPU[0]		: Memory Activity: N/A
GPU[0]		: Not supported on the given system
================================== End of ROCm SMI Log ===================================
$ /home/aup/envs/robocasa365-rocm-py312/bin/python /path/to/every-embodied/mujoco_pnp/train_model.py --config_path /home/aup/amd-notebook-work/course/outputs/configs/pi0_smoke.yaml
当前只预览命令。确认配置后，把对应 RUN_* 开关改为 True。


2-step smoke 只证明数据加载、模型构造、forward/backward、optimizer step 和 checkpoint 写出可用；它不证明模型收敛，更不能替代 MuJoCo closed-loop 成功率。


## Checkpoint 5：启动正式训练


In [9]:
run_training(full_config_path, enabled=RUN_FULL_TRAIN)


$ /home/aup/envs/robocasa365-rocm-py312/bin/python /path/to/every-embodied/mujoco_pnp/train_model.py --config_path /home/aup/amd-notebook-work/course/outputs/configs/pi0_full.yaml
当前只预览命令。确认配置后，把对应 RUN_* 开关改为 True。


pi_0 的训练 loss 下降并不保证闭环抓取成功。小数据接触任务尤其容易出现 gripper 时序和误差累积问题。raw policy、learned auxiliary head 和带脚手架的 hybrid 结果必须分开报告。


常见恢复顺序：显存不足先减小 `batch_size`；出现 DataLoader worker / pickle 错误时把 YAML 中的 `num_workers` 改为 `0`；下载报 `401/403` 时检查账号权限和私密 `HF_TOKEN`；写 checkpoint 失败先检查 `$MODEL_ROOT` 所在磁盘。ROCm 的 MIOpen 数据库 warning 如果没有伴随训练退出可以记录后继续观察，出现 kernel 退出、NaN 或进程消失则必须停止长训练并回到 smoke。


## Checkpoint 6：定位 checkpoint 并检查资源


In [10]:
show_rocm_resources()
if (FULL_OUTPUT / "checkpoints").exists():
    checkpoints = sorted((FULL_OUTPUT / "checkpoints").glob("*/pretrained_model"))
    print("可用 checkpoint：")
    for path in checkpoints:
        print(" -", path)
else:
    print("正式训练尚未运行：", FULL_OUTPUT)


$ rocm-smi --showuse --showmemuse --showtemp


============================ ROCm System Management Interface ============================
====================================== Temperature =======================================
GPU[0]		: Temperature (Sensor edge) (C): 35.0
=================================== % time GPU is busy ===================================
GPU[0]		: GPU use (%): 0
=================================== Current Memory Use ===================================
GPU[0]		: GPU Memory Allocated (VRAM%): 0
GPU[0]		: Memory Activity: N/A
GPU[0]		: Not supported on the given system
================================== End of ROCm SMI Log ===================================
正式训练尚未运行： /path/to/every-embodied/mujoco_pnp/ckpt/pi0_rocm_full


训练完成后不要只挑 loss 最低的节点。下一步进入 `11_mujoco_closed_loop_deploy.ipynb`，用固定 seed、严格 `physical_success` 和视频复核比较中间 checkpoint 与最终 checkpoint。


## 已完成训练的日志快照


In [11]:
progress = json.loads(
    (ASSET_DIR / "training_progress_snapshot.json").read_text(encoding="utf-8")
)
records = [row for row in progress["records"] if row["model"] == 'pi0']
for row in records:
    ratio = row["current_step"] / max(1, row["total_steps"])
    filled = round(36 * ratio)
    bar = "#" * filled + "." * (36 - filled)
    print(f'{row["model"]:>18} [{bar}] {row["current_step"]}/{row["total_steps"]}')
    print("stage:", row["stage"])
    print("closed-loop physical_success:", row["physical_success"])
    print("note:", row["note"])


               pi0 [####################################] 1500/1500
stage: public raw baseline
closed-loop physical_success: 0/20
note: loss 从 step500 约 0.179 降到 step1500 约 0.046，但 raw closed-loop 没有成功


![AMD ROCm 历史训练进度与闭环结果](../../assets/training_progress_overview.png)

                这张图来自已经完成的 AMD 实验日志，不是假装本次打开 Notebook 后重新跑完的训练。上面的 smoke/full 单元提供真实启动代码；运行长训练时，终端进度会继续写入输出目录，完成后再用 11 的闭环评估决定 checkpoint 是否可用。


## 在 Notebook 中跟踪后台训练日志


In [12]:
def tail_training_log(log_path, lines=25):
    log_path = Path(log_path)
    if not log_path.exists():
        print("训练日志尚不存在：", log_path)
        return
    content = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n".join(content[-lines:]))


TRAIN_LOG = Path(os.environ.get("TRAIN_LOG", FULL_OUTPUT / "train.log"))
print("训练运行后可重复执行：tail_training_log(TRAIN_LOG)")
tail_training_log(TRAIN_LOG)


训练运行后可重复执行：tail_training_log(TRAIN_LOG)
训练日志尚不存在： /path/to/every-embodied/mujoco_pnp/ckpt/pi0_rocm_full/train.log
